# Time Series Forecasting: Traditional vs ML vs Deep Learning

A comprehensive comparison of three forecasting paradigms on the **Beijing PM2.5 Air Quality** dataset (~44K hourly observations, 2010–2014).

| Part | Approach | Models |
|------|----------|--------|
| **A** | Traditional | ARIMA, GARCH volatility analysis |
| **B** | Machine Learning | Linear Regression, Random Forest, XGBoost |
| **C** | Deep Learning | LSTM (basic & regularized), 1D-CNN |

**Goal:** Forecast hourly PM2.5 concentration. Train on pre-2014 data, test on 2014.

> **Kaggle setup:** Enable GPU accelerator (Settings → Accelerator → GPU) for faster DL training.


## 1) Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error

import xgboost as xgb
from arch import arch_model

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Conv1D, Flatten, GRU
from tensorflow.keras.callbacks import EarlyStopping

sns.set_theme(style='whitegrid')
tf.random.set_seed(42)
np.random.seed(42)

print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))


## 2) Data Loading

In [ ]:
from pathlib import Path

candidates = [
    Path('/kaggle/input/datasets/trongnghia7171/beijing-air-quality/beijing_air_quality.csv'),
    Path('/kaggle/input/beijing-air-quality/beijing_air_quality.csv'),
    Path('data/beijing_air_quality.csv'),
]

raw_path = None
for p in candidates:
    if p.exists():
        raw_path = p
        break

if raw_path is None:
    raise FileNotFoundError(
        'Dataset not found. Add the Beijing Air Quality dataset on Kaggle '
        'or place beijing_air_quality.csv in data/.'
    )

df = pd.read_csv(raw_path, parse_dates=['datetime']).set_index('datetime').sort_index()

print('Loaded from:', raw_path)
print('Shape:', df.shape)
print('Date range:', df.index.min(), '→', df.index.max())
print('Columns:', list(df.columns))
df.head()


## 3) Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

df['pm25'].plot(ax=axes[0], linewidth=0.3, alpha=0.7, color='steelblue')
axes[0].set_title('Hourly PM2.5 (2010 – 2014)')
axes[0].set_ylabel('µg/m³')

df.groupby(df.index.hour)['pm25'].mean().plot(ax=axes[1], marker='o', color='coral')
axes[1].set_title('Average PM2.5 by Hour of Day')
axes[1].set_xlabel('Hour')

df.groupby(df.index.month)['pm25'].mean().plot(kind='bar', ax=axes[2], color='mediumseagreen')
axes[2].set_title('Average PM2.5 by Month')
axes[2].set_xlabel('Month')

plt.tight_layout()
plt.show()


In [ ]:
print('=== PM2.5 Summary Statistics ===')
print(df['pm25'].describe().round(2))
print(f'\nMissing values per column:\n{df.isnull().sum()}')


**Observations:**
- PM2.5 shows strong daily seasonality (higher at night) and annual seasonality (winter peaks).
- The series is highly variable with extreme spikes — good candidate for volatility analysis.
- These patterns suggest that models capturing temporal dependencies should outperform simple baselines.


---
# Part A: Traditional Time Series Models
---


## A1) ARIMA Baseline

ARIMA works on **univariate** series and assumes constant variance.  
We apply it to **daily-averaged** PM2.5 (hourly ARIMA is computationally prohibitive and not designed for 44K points).


In [ ]:
daily = df['pm25'].resample('D').mean().dropna()
daily_train = daily.loc[daily.index < '2014']
daily_test = daily.loc[daily.index >= '2014']

print(f'Daily train: {len(daily_train)} days')
print(f'Daily test:  {len(daily_test)} days')

adf_stat, adf_p, *_ = adfuller(daily_train)
print(f'\nADF test on daily train: stat={adf_stat:.4f}, p={adf_p:.4f}')
print('Stationary:', 'Yes' if adf_p < 0.05 else 'No — differencing needed')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(daily_train.diff().dropna(), lags=40, ax=axes[0])
axes[0].set_title('ACF of Differenced Daily PM2.5')
plot_pacf(daily_train.diff().dropna(), lags=40, ax=axes[1])
axes[1].set_title('PACF of Differenced Daily PM2.5')
plt.tight_layout()
plt.show()


In [ ]:
arima_model = ARIMA(daily_train, order=(2, 1, 1))
arima_fit = arima_model.fit()
print(arima_fit.summary())


In [ ]:
def calc_rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

arima_fc = arima_fit.forecast(steps=len(daily_test))
arima_mae = mean_absolute_error(daily_test, arima_fc)
arima_rmse = calc_rmse(daily_test, arima_fc)

print(f'ARIMA(2,1,1) on daily PM2.5:')
print(f'  MAE  = {arima_mae:.2f}')
print(f'  RMSE = {arima_rmse:.2f}')

fig, ax = plt.subplots(figsize=(14, 4))
daily_test.plot(ax=ax, label='Actual', color='black')
arima_fc.plot(ax=ax, label='ARIMA Forecast', color='red', linestyle='--')
ax.set_title('ARIMA Daily Forecast vs Actual (2014)')
ax.legend()
plt.tight_layout()
plt.show()


## A2) GARCH Volatility Analysis

ARIMA assumes constant variance, but PM2.5 residuals show **volatility clustering** — periods of high variance follow high variance.  
GARCH captures this heteroscedastic behavior.


In [ ]:
resid = arima_fit.resid.dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
resid.plot(ax=axes[0], title='ARIMA Residuals', color='steelblue')
(resid ** 2).plot(ax=axes[1], title='Squared Residuals (volatility clustering?)', color='coral')
plt.tight_layout()
plt.show()


In [ ]:
garch = arch_model(resid, vol='Garch', p=1, q=1, mean='Zero', rescale=True)
garch_fit = garch.fit(disp='off')
print(garch_fit.summary())


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
cond_vol = garch_fit.conditional_volatility
cond_vol.index = resid.index[:len(cond_vol)]
cond_vol.plot(ax=ax, color='darkorange', label='Conditional Volatility')
ax.set_title('GARCH(1,1) Conditional Volatility of ARIMA Residuals')
ax.set_ylabel('Volatility')
ax.legend()
plt.tight_layout()
plt.show()


**Interpretation:**
- Significant ARCH (α₁) and GARCH (β₁) coefficients confirm volatility clustering.
- GARCH doesn't improve point forecasts, but it provides **prediction intervals** — critical for risk-sensitive applications like health advisories.
- Traditional models are limited: ARIMA handles only univariate daily data and cannot leverage weather covariates.


---
# Part B: Machine Learning Models
---

ML models work on **hourly** data (full granularity) and can leverage multiple weather features.  
The key step is converting the time series into a **supervised learning** tabular format.


## B1) Supervised Learning Transformation

We create lag features, rolling statistics, and calendar variables to capture temporal patterns:


In [ ]:
TARGET = 'pm25'
FEATURE_COLS_RAW = ['pm25', 'temperature', 'pressure', 'dewpoint', 'wind_speed']

ml_df = df[FEATURE_COLS_RAW].copy()

for lag in [1, 2, 3, 6, 12, 24]:
    ml_df[f'lag_{lag}'] = ml_df[TARGET].shift(lag)

for window in [6, 24]:
    ml_df[f'rolling_mean_{window}'] = ml_df[TARGET].shift(1).rolling(window).mean()
    ml_df[f'rolling_std_{window}'] = ml_df[TARGET].shift(1).rolling(window).std()

ml_df['hour'] = ml_df.index.hour
ml_df['day_of_week'] = ml_df.index.dayofweek
ml_df['month'] = ml_df.index.month
ml_df['is_weekend'] = (ml_df.index.dayofweek >= 5).astype(int)

ml_df = ml_df.dropna()
print('Tabular dataset shape:', ml_df.shape)
ml_df.head()


## B2) Train ML Models

In [ ]:
feature_cols = [c for c in ml_df.columns if c != TARGET]

ml_train = ml_df.loc[ml_df.index < '2014']
ml_test = ml_df.loc[ml_df.index >= '2014']

X_train_ml, y_train_ml = ml_train[feature_cols], ml_train[TARGET]
X_test_ml, y_test_ml = ml_test[feature_cols], ml_test[TARGET]

print('ML Train:', X_train_ml.shape, '  Test:', X_test_ml.shape)


In [ ]:
ml_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=15, n_jobs=-1, random_state=42
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        n_jobs=-1, random_state=42, verbosity=0
    ),
}

ml_results = []
ml_predictions = {}

for name, model in ml_models.items():
    model.fit(X_train_ml, y_train_ml)
    y_pred = model.predict(X_test_ml)
    ml_predictions[name] = y_pred
    ml_results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test_ml, y_pred),
        'RMSE': calc_rmse(y_test_ml, y_pred),
    })

ml_results_df = pd.DataFrame(ml_results).set_index('Model').sort_values('RMSE')
ml_results_df


## B3) Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name in zip(axes, ['Random Forest', 'XGBoost']):
    model = ml_models[name]
    imp = pd.Series(model.feature_importances_, index=feature_cols).nlargest(15)
    imp.plot(kind='barh', ax=ax, title=f'{name} — Top 15 Features')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()


**Observations:**
- Lag-1 and recent rolling-mean features dominate — strong short-term autocorrelation.
- Weather features (temperature, pressure, dewpoint) provide additional predictive signal.
- Calendar features (hour, month) capture seasonal patterns the tree models can split on.


## B4) Temporal Validation (TimeSeriesSplit)

Standard k-fold CV leaks future information. `TimeSeriesSplit` preserves temporal ordering.


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

cv_results = {}
for name, model in ml_models.items():
    scores = cross_val_score(
        model, X_train_ml, y_train_ml,
        cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1,
    )
    cv_results[name] = -scores

cv_df = pd.DataFrame(cv_results)
print(cv_df.round(2))
print('\nMean MAE across folds:')
print(cv_df.mean().round(2))

ax = cv_df.plot(kind='bar', figsize=(10, 5))
ax.set_title('MAE per TimeSeriesSplit Fold')
ax.set_xlabel('Fold')
ax.set_ylabel('MAE')
plt.tight_layout()
plt.show()


---
# Part C: Deep Learning Models
---

DL models learn temporal patterns directly from **raw sequences** — no manual feature engineering needed.  
They require a specific pipeline: **split → scale → window into 3D tensors → train**.


## C1) DL Preprocessing Pipeline

1. **Split** chronologically (same as ML: pre-2014 train, 2014 test).
2. **Scale** features to [0, 1] — fit on training data only.
3. **Window** the scaled sequence into 3D tensors `(samples, time_steps, features)` with a 24-hour lookback.


In [ ]:
DL_FEATURES = ['pm25', 'temperature', 'pressure', 'dewpoint', 'wind_speed']
LOOKBACK = 24
EPOCHS = 20
BATCH_SIZE = 64

dl_series = df[DL_FEATURES].dropna().copy()

dl_train_raw = dl_series.loc[dl_series.index < '2014']
dl_test_raw = dl_series.loc[dl_series.index >= '2014']

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(dl_train_raw)
test_scaled = scaler.transform(dl_test_raw)

target_idx = DL_FEATURES.index(TARGET)

print(f'DL Train: {dl_train_raw.shape}  Test: {dl_test_raw.shape}')
print(f'Target column index: {target_idx}')


In [ ]:
def create_sequences(data, lookback, target_idx, horizon=1):
    """Convert 2D scaled array into (X, y) for DL models."""
    X, y = [], []
    for i in range(lookback, len(data) - horizon + 1):
        X.append(data[i - lookback:i])
        if horizon == 1:
            y.append(data[i, target_idx])
        else:
            y.append(data[i:i + horizon, target_idx])
    return np.array(X), np.array(y)

X_train_dl, y_train_dl = create_sequences(train_scaled, LOOKBACK, target_idx)
X_test_dl, y_test_dl = create_sequences(test_scaled, LOOKBACK, target_idx)

print(f'X_train: {X_train_dl.shape}  y_train: {y_train_dl.shape}')
print(f'X_test:  {X_test_dl.shape}   y_test:  {y_test_dl.shape}')
print(f'\nTensor: ({X_train_dl.shape[0]} samples, {X_train_dl.shape[1]} time steps, {X_train_dl.shape[2]} features)')


## C2) LSTM — Basic

LSTMs maintain a cell state that captures long-term dependencies across the 24-hour lookback window.


In [ ]:
lstm_basic = Sequential([
    LSTM(64, activation='relu', input_shape=(LOOKBACK, len(DL_FEATURES))),
    Dense(1),
])
lstm_basic.compile(optimizer='adam', loss='mse')

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

lstm_basic_hist = lstm_basic.fit(
    X_train_dl, y_train_dl,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop],
    shuffle=False, verbose=1,
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lstm_basic_hist.history['loss'], label='Train')
ax.plot(lstm_basic_hist.history['val_loss'], label='Validation')
ax.set_title('LSTM (basic) Training Curve')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout(); plt.show()


## C3) LSTM — Regularized (Stacked + Dropout)

Adding a second LSTM layer and Dropout reduces overfitting.


In [ ]:
lstm_reg = Sequential([
    LSTM(64, activation='relu', return_sequences=True, input_shape=(LOOKBACK, len(DL_FEATURES))),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dropout(0.2),
    Dense(1),
])
lstm_reg.compile(optimizer='adam', loss='mse')

lstm_reg_hist = lstm_reg.fit(
    X_train_dl, y_train_dl,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    shuffle=False, verbose=1,
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lstm_reg_hist.history['loss'], label='Train')
ax.plot(lstm_reg_hist.history['val_loss'], label='Validation')
ax.set_title('LSTM (regularized) Training Curve')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout(); plt.show()


## C4) GRU

GRU is a lighter alternative to LSTM — fewer parameters, often similar performance, and faster training.


In [ ]:
gru_model = Sequential([
    GRU(64, activation='relu', input_shape=(LOOKBACK, len(DL_FEATURES))),
    Dense(1),
])
gru_model.compile(optimizer='adam', loss='mse')

gru_hist = gru_model.fit(
    X_train_dl, y_train_dl,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    shuffle=False, verbose=1,
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(gru_hist.history['loss'], label='Train')
ax.plot(gru_hist.history['val_loss'], label='Validation')
ax.set_title('GRU Training Curve')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout(); plt.show()


## C5) 1D-CNN

Convolutional filters slide across the time axis to extract local temporal patterns.  
Often faster than LSTMs and surprisingly effective.


In [ ]:
cnn_model = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(LOOKBACK, len(DL_FEATURES))),
    Conv1D(32, kernel_size=3, activation='relu'),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(1),
])
cnn_model.compile(optimizer='adam', loss='mse')

cnn_hist = cnn_model.fit(
    X_train_dl, y_train_dl,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    shuffle=False, verbose=1,
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cnn_hist.history['loss'], label='Train')
ax.plot(cnn_hist.history['val_loss'], label='Validation')
ax.set_title('1D-CNN Training Curve')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout(); plt.show()


## C6) DL Post-processing & Evaluation

Predictions are in [0, 1] scaled space — we inverse-transform back to original units.


In [ ]:
def inverse_target(scaled_preds, scaler, target_idx, n_features):
    """Inverse-transform only the target column."""
    dummy = np.zeros((len(scaled_preds), n_features))
    dummy[:, target_idx] = scaled_preds.flatten()
    return scaler.inverse_transform(dummy)[:, target_idx]

n_feat = len(DL_FEATURES)
y_test_real = inverse_target(y_test_dl, scaler, target_idx, n_feat)

dl_models = {
    'LSTM (basic)': lstm_basic,
    'LSTM (regularized)': lstm_reg,
    'GRU': gru_model,
    '1D-CNN': cnn_model,
}

dl_results = []
dl_preds = {}

for name, model in dl_models.items():
    preds_scaled = model.predict(X_test_dl, verbose=0)
    preds_real = inverse_target(preds_scaled, scaler, target_idx, n_feat)
    dl_preds[name] = preds_real
    dl_results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test_real, preds_real),
        'RMSE': calc_rmse(y_test_real, preds_real),
    })

dl_results_df = pd.DataFrame(dl_results).set_index('Model').sort_values('RMSE')
dl_results_df


In [ ]:
week_len = 24 * 7

plt.figure(figsize=(14, 5))
plt.plot(y_test_real[:week_len], label='Actual', color='black', linewidth=2)
for name, preds in dl_preds.items():
    plt.plot(preds[:week_len], label=name, alpha=0.8)
plt.title('DL Forecasts — First Week of 2014 Test Set')
plt.xlabel('Hour'); plt.ylabel('PM2.5 (µg/m³)'); plt.legend()
plt.tight_layout(); plt.show()


## C7) Multi-step Forecasting (6 hours ahead)

Instead of predicting just t+1, we predict the next **6 hours** simultaneously.


In [ ]:
HORIZON = 6

X_train_ms, y_train_ms = create_sequences(train_scaled, LOOKBACK, target_idx, horizon=HORIZON)
X_test_ms, y_test_ms = create_sequences(test_scaled, LOOKBACK, target_idx, horizon=HORIZON)

print(f'Multi-step shapes — X_train: {X_train_ms.shape}, y_train: {y_train_ms.shape}')

lstm_ms = Sequential([
    LSTM(64, activation='relu', input_shape=(LOOKBACK, len(DL_FEATURES))),
    Dense(HORIZON),
])
lstm_ms.compile(optimizer='adam', loss='mse')

lstm_ms.fit(
    X_train_ms, y_train_ms,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    shuffle=False, verbose=1,
)


In [ ]:
ms_preds = lstm_ms.predict(X_test_ms, verbose=0)

ms_mae = []
for h in range(HORIZON):
    yt = inverse_target(y_test_ms[:, h], scaler, target_idx, n_feat)
    yp = inverse_target(ms_preds[:, h], scaler, target_idx, n_feat)
    ms_mae.append(mean_absolute_error(yt, yp))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, HORIZON + 1), ms_mae, color='steelblue')
ax.set_title('Multi-step LSTM: MAE by Forecast Horizon')
ax.set_xlabel('Hours Ahead'); ax.set_ylabel('MAE (PM2.5)')
plt.tight_layout(); plt.show()

print('MAE per step:', [f'{m:.2f}' for m in ms_mae])


**Observation:** MAE increases with horizon — predicting 6 hours ahead is harder than 1 hour. This is the fundamental accuracy–horizon trade-off in multi-step forecasting.


---
# Grand Comparison: Traditional vs ML vs DL
---


In [ ]:
comparison = pd.concat([
    ml_results_df.assign(Paradigm='ML'),
    dl_results_df.assign(Paradigm='DL'),
])

comparison.loc['ARIMA (daily)'] = {
    'MAE': arima_mae, 'RMSE': arima_rmse, 'Paradigm': 'Traditional'
}

comparison = comparison.sort_values('RMSE')
comparison.style.format({'MAE': '{:.2f}', 'RMSE': '{:.2f}'}).highlight_min(
    subset=['MAE', 'RMSE'], axis=0, color='#d4edda'
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Traditional': '#e74c3c', 'ML': '#3498db', 'DL': '#2ecc71'}
comp_plot = comparison.reset_index()

for metric, ax in zip(['MAE', 'RMSE'], axes):
    bars = ax.barh(
        comp_plot['Model'], comp_plot[metric],
        color=[colors.get(p, 'gray') for p in comp_plot['Paradigm']]
    )
    ax.set_title(metric)
    ax.set_xlabel(metric)

handles = [plt.Rectangle((0,0),1,1, color=c) for c in colors.values()]
fig.legend(handles, colors.keys(), loc='upper right', fontsize=10, title='Paradigm')
plt.tight_layout()
plt.show()


## Forecast Overlay — One Week

In [ ]:
# Align ML and DL predictions on the same hourly test period
# DL test starts LOOKBACK hours into the test set due to windowing
dl_test_index = dl_test_raw.index[LOOKBACK:]
ml_test_index = ml_test.index

common_start = max(dl_test_index[0], ml_test_index[0])
common_end = common_start + pd.Timedelta(days=7)

fig, ax = plt.subplots(figsize=(14, 6))

# Actual
actual_slice = df['pm25'].loc[common_start:common_end]
ax.plot(actual_slice.index, actual_slice.values, color='black', linewidth=2, label='Actual')

# ML: XGBoost
xgb_idx = (ml_test_index >= common_start) & (ml_test_index <= common_end)
ax.plot(ml_test_index[xgb_idx], ml_predictions['XGBoost'][xgb_idx],
        label='XGBoost', alpha=0.8, linestyle='--')

# DL: best LSTM and CNN
for name in ['LSTM (regularized)', '1D-CNN']:
    dl_idx = (dl_test_index >= common_start) & (dl_test_index <= common_end)
    ax.plot(dl_test_index[dl_idx], dl_preds[name][dl_idx[:len(dl_preds[name])]],
            label=name, alpha=0.8)

ax.set_title('PM2.5 Forecast — First Week of Test Set')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.legend()
plt.tight_layout()
plt.show()


---
## Discussion & Conclusions

### Performance Summary

| Paradigm | Best Model | Strengths | Weaknesses |
|----------|-----------|-----------|------------|
| **Traditional** | ARIMA(2,1,1) | Interpretable, no tuning, confidence intervals | Univariate only, daily granularity, high MAE |
| **ML** | XGBoost | Fast, handles features, interpretable importance | Needs manual feature engineering, one-step only |
| **DL** | LSTM / CNN | Learns from raw sequences, multi-step native | Needs GPU, less interpretable, more data-hungry |

### When to Use Which?

- **ARIMA/GARCH:** Small datasets, univariate, when interpretability and prediction intervals matter (e.g., risk assessment).
- **XGBoost/RF:** Medium datasets, when you have domain knowledge to engineer features, when speed and interpretability are priorities.
- **LSTM/GRU/CNN:** Large datasets (10K+ points), complex multi-variate sequences, multi-step forecasting, when GPU is available.

### Key Takeaways

1. **Feature engineering** (lags, rolling stats) is the single biggest performance booster for ML models — it encodes temporal structure explicitly.
2. **DL models** learn equivalent temporal patterns automatically from raw sequences, but need more data and compute.
3. **GARCH** complements any approach by adding uncertainty quantification to residuals.
4. **No single paradigm wins everywhere** — the best choice depends on data size, compute budget, interpretability needs, and forecast horizon.

### Further Reading
- **Darts** / **NeuralProphet** / **GluonTS**: Libraries that simplify DL time series workflows.
- **Temporal Fusion Transformer (TFT)**: State-of-the-art DL architecture for interpretable multi-horizon forecasting.
- **Conformal prediction**: Framework for adding valid prediction intervals to any model.
